In [ ]:
#CONNECT GPU

MODULE 1

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install datasets transformers torch accelerate -q

In [ ]:
import os
import random
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import RobertaTokenizer
import json
from collections import defaultdict

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print("✓ Libraries imported and seed set")

✓ Libraries imported and seed set


In [ ]:
# Directory in Google Drive to save datasets
SAVE_DIR = '/content/drive/MyDrive/FYP_GradientFree_LoRA/'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, 'datasets'), exist_ok=True)

print(f"✓ Datasets will be saved to: {SAVE_DIR}")

✓ Datasets will be saved to: /content/drive/MyDrive/FYP_GradientFree_LoRA/


In [ ]:
# Patterns and verbalizers for each dataset
DATASET_CONFIG = {
    'sst2': {
        'dataset_name': 'sst2',
        'dataset_config': None,
        'text_column': 'sentence',
        'label_column': 'label',
        'pattern': lambda text: f"{text} It was <mask>.",
        'verbalizers': {0: "bad", 1: "great"},
        'num_classes': 2,
        'use_validation_as_test': True  # SST-2 test set has no labels
    },
    'yelp_polarity': {
        'dataset_name': 'yelp_polarity',
        'dataset_config': None,
        'text_column': 'text',
        'label_column': 'label',
        'pattern': lambda text: f"{text} It was <mask>.",
        'verbalizers': {0: "bad", 1: "great"},
        'num_classes': 2,
        'use_validation_as_test': False
    },
    'ag_news': {
        'dataset_name': 'ag_news',
        'dataset_config': None,
        'text_column': 'text',
        'label_column': 'label',
        'pattern': lambda text: f"<mask> News {text}.",
        'verbalizers': {0: "World", 1: "Sports", 2: "Business", 3: "Tech"},
        'num_classes': 4,
        'use_validation_as_test': False
    },
    'dbpedia_14': {
        'dataset_name': 'dbpedia_14',
        'dataset_config': None,
        'text_column': 'content',
        'label_column': 'label',
        'pattern': lambda text: f"Category: <mask> {text}",
        'verbalizers': {
            0: "Company", 1: "Education", 2: "Artist", 3: "Athlete",
            4: "Office", 5: "Transportation", 6: "Building", 7: "Natural",
            8: "Village", 9: "Animal", 10: "Plant", 11: "Album",
            12: "Film", 13: "Written"
        },
        'num_classes': 14,
        'use_validation_as_test': False
    }
}

print("✓ Dataset configurations defined")
print(f"Datasets to process: {list(DATASET_CONFIG.keys())}")

✓ Dataset configurations defined
Datasets to process: ['sst2', 'yelp_polarity', 'ag_news', 'dbpedia_14']


In [ ]:
# Initializing RoBERTa tokenizer
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
print("✓ RoBERTa tokenizer loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✓ RoBERTa tokenizer loaded


In [ ]:
def create_few_shot_split(dataset, n_samples_per_class, num_classes, label_column, seed=42):

    random.seed(seed)
    np.random.seed(seed)

    # Group samples by class
    class_samples = defaultdict(list)
    for idx, example in enumerate(dataset):
        label = example[label_column]
        class_samples[label].append(idx)

    # Sample n instances per class for train and dev
    train_indices = []
    dev_indices = []

    for label in range(num_classes):
        if label not in class_samples:
            print(f"Warning: No samples found for class {label}")
            continue

        samples = class_samples[label]
        random.shuffle(samples)


        if len(samples) < 2 * n_samples_per_class:
            print(f"Warning: Class {label} has only {len(samples)} samples, need {2*n_samples_per_class}")
            n = min(n_samples_per_class, len(samples) // 2)
        else:
            n = n_samples_per_class

        train_indices.extend(samples[:n])
        dev_indices.extend(samples[n:2*n])

    train_split = dataset.select(train_indices)
    dev_split = dataset.select(dev_indices)

    return train_split, dev_split

print("✓ Few-shot split function defined")

✓ Few-shot split function defined


In [ ]:
def apply_pattern_and_verbalizer(example, config):
    text = example[config['text_column']]
    label = example[config['label_column']]

    max_text_length = 400
    if len(text) > max_text_length:
        text = text[:max_text_length]

    # Apply pattern
    mlm_text = config['pattern'](text)

    # Getting verbalizer for the label
    verbalizer = config['verbalizers'][label]

    return {
        'mlm_text': mlm_text,
        'text': text,
        'label': label,
        'verbalizer': verbalizer
    }

print("✓ Pattern application function defined")

✓ Pattern application function defined


In [ ]:
def process_dataset(dataset_key, n_shot=16, save=True):
    """
    Process a single dataset: load, create few-shot splits, apply patterns
    """
    print(f"\n{'='*60}")
    print(f"Processing {dataset_key} with {n_shot}-shot setting")
    print(f"{'='*60}")

    config = DATASET_CONFIG[dataset_key]

    # Loading dataset
    print("Loading dataset")
    if config['dataset_config']:
        dataset = load_dataset(config['dataset_name'], config['dataset_config'])
    else:
        dataset = load_dataset(config['dataset_name'])


    if config.get('use_validation_as_test', False):
        print("Using validation set as test set (original test set has no labels)")
        train_full = dataset['train']
        test_full = dataset['validation']
    else:
        train_full = dataset['train']
        test_full = dataset['test']


    valid_labels = set(config['verbalizers'].keys())
    print(f"Valid labels: {valid_labels}")

    train_full = train_full.filter(lambda x: x[config['label_column']] in valid_labels)
    test_full = test_full.filter(lambda x: x[config['label_column']] in valid_labels)

    print(f"Original train size (after filtering): {len(train_full)}")
    print(f"Original test size (after filtering): {len(test_full)}")

    # Creating few-shot splits
    print(f"\nCreating {n_shot}-shot splits...")
    train_split, dev_split = create_few_shot_split(
        train_full,
        n_samples_per_class=n_shot,
        num_classes=config['num_classes'],
        label_column=config['label_column'],
        seed=42
    )

    print(f"Few-shot train size: {len(train_split)}")
    print(f"Few-shot dev size: {len(dev_split)}")

    # Apply patterns and verbalizers
    print("\nApplying patterns and verbalizers...")
    train_processed = train_split.map(
        lambda x: apply_pattern_and_verbalizer(x, config),
        batched=False
    )
    dev_processed = dev_split.map(
        lambda x: apply_pattern_and_verbalizer(x, config),
        batched=False
    )
    test_processed = test_full.map(
        lambda x: apply_pattern_and_verbalizer(x, config),
        batched=False
    )

    # Create DatasetDict
    processed_dataset = DatasetDict({
        'train': train_processed,
        'validation': dev_processed,
        'test': test_processed
    })


    print("\n--- Sample from training set ---")
    print(f"Original text: {train_processed[0]['text'][:100]}...")
    print(f"MLM text: {train_processed[0]['mlm_text'][:150]}...")
    print(f"Label: {train_processed[0]['label']}")
    print(f"Verbalizer: {train_processed[0]['verbalizer']}")

    # Save to Google Drive
    if save:
        save_path = os.path.join(SAVE_DIR, 'datasets', f'{dataset_key}_{n_shot}shot')
        print(f"\nSaving to: {save_path}")
        processed_dataset.save_to_disk(save_path)


        config_save = {
            'pattern': str(config['pattern']('TEXT_PLACEHOLDER')),
            'verbalizers': config['verbalizers'],
            'n_shot': n_shot,
            'num_classes': config['num_classes']
        }
        with open(os.path.join(save_path, 'config.json'), 'w') as f:
            json.dump(config_save, f, indent=2)

    print(f"\n✓ {dataset_key} processing complete!")

    return processed_dataset

print("✓ Main processing function defined")

✓ Main processing function defined


In [ ]:
# Setting the number of samples per class for few-shot learning
n_shot = 16

print(f"✓ Using {n_shot}-shot setting")

✓ Using 16-shot setting


In [ ]:
# Process SST-2 (Sentiment Analysis)
print("\n" + "="*60)
print("PROCESSING SST-2")
print("="*60)

sst2_dataset = process_dataset('sst2', n_shot=n_shot)


PROCESSING SST-2

Processing sst2 with 16-shot setting
Loading dataset


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

... (10 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/872 [00:00<?, ? examples/s]


✓ sst2 processing complete!


In [ ]:
# Process Yelp Polarity (Sentiment Analysis)
print("\n" + "="*60)
print("PROCESSING YELP POLARITY")
print("="*60)

yelp_dataset = process_dataset('yelp_polarity', n_shot=n_shot)


PROCESSING YELP POLARITY

Processing yelp_polarity with 16-shot setting
Loading dataset


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

... (8 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/38000 [00:00<?, ? examples/s]


✓ yelp_polarity processing complete!


In [ ]:
# Process AG News (Topic Classification)
print("\n" + "="*60)
print("PROCESSING AG NEWS")
print("="*60)

agnews_dataset = process_dataset('ag_news', n_shot=n_shot)


PROCESSING AG NEWS

Processing ag_news with 16-shot setting
Loading dataset


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

... (8 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/64 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/64 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7600 [00:00<?, ? examples/s]


✓ ag_news processing complete!


In [ ]:
# Process DBPedia (Topic Classification)
print("\n" + "="*60)
print("PROCESSING DBPEDIA")
print("="*60)

dbpedia_dataset = process_dataset('dbpedia_14', n_shot=n_shot)


PROCESSING DBPEDIA

Processing dbpedia_14 with 16-shot setting
Loading dataset


README.md: 0.00B [00:00, ?B/s]

dbpedia_14/train-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

dbpedia_14/test-00000-of-00001.parquet:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

... (8 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/70000 [00:00<?, ? examples/s]


✓ dbpedia_14 processing complete!


In [ ]:
# Print summary of all processed datasets
print("\n" + "="*60)
print("PROCESSING COMPLETE - SUMMARY")
print("="*60)

processed_datasets = {
    'sst2': sst2_dataset,
    'yelp_polarity': yelp_dataset,
    'ag_news': agnews_dataset,
    'dbpedia_14': dbpedia_dataset
}

for name, ds in processed_datasets.items():
    print(f"\n{name.upper()}:")
    print(f"  Train: {len(ds['train'])} samples")
    print(f"  Validation: {len(ds['validation'])} samples")
    print(f"  Test: {len(ds['test'])} samples")

print(f"\n✓ All datasets saved to: {SAVE_DIR}datasets/")
print("\n--- How to load datasets later ---")
print("from datasets import load_from_disk")
print("dataset = load_from_disk('/content/drive/MyDrive/FYP_GradientFree_LoRA/datasets/sst2_16shot')")


PROCESSING COMPLETE - SUMMARY

SST2:
  Train: 32 samples
  Validation: 32 samples
  Test: 872 samples

YELP_POLARITY:
  Train: 32 samples
  Validation: 32 samples
  Test: 38000 samples

AG_NEWS:
  Train: 64 samples
  Validation: 64 samples
  Test: 7600 samples

DBPEDIA_14:
  Train: 224 samples
  Validation: 224 samples
  Test: 70000 samples

✓ All datasets saved to: /content/drive/MyDrive/FYP_GradientFree_LoRA/datasets/

--- How to load datasets later ---
from datasets import load_from_disk
dataset = load_from_disk('/content/drive/MyDrive/FYP_GradientFree_LoRA/datasets/sst2_16shot')


In [ ]:
import pandas as pd
from IPython.display import display


def display_dataset_as_table(dataset_name, dataset, split='train', max_rows=None):
    print(f"\n{'='*100}")
    print(f"{dataset_name.upper()} - {split.upper()} SPLIT")
    print(f"{'='*100}\n")


    df = dataset[split].to_pandas()


    df['text_preview'] = df['text'].str[:100] + '...'
    df['mlm_text_preview'] = df['mlm_text'].str[:150] + '...'


    display_cols = ['label', 'verbalizer', 'text_preview', 'mlm_text_preview']
    df_display = df[display_cols]


    pd.set_option('display.max_rows', max_rows)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)

    print(f"Total records: {len(df)}\n")
    display(df_display)

    return df

# Display all datasets

# SST-2
sst2_train_df = display_dataset_as_table('sst2', sst2_dataset, 'train', max_rows=None)
sst2_val_df = display_dataset_as_table('sst2', sst2_dataset, 'validation', max_rows=None)
sst2_test_df = display_dataset_as_table('sst2', sst2_dataset, 'test', max_rows=50)  # Limit test to 50 rows

# Yelp Polarity
yelp_train_df = display_dataset_as_table('yelp_polarity', yelp_dataset, 'train', max_rows=None)
yelp_val_df = display_dataset_as_table('yelp_polarity', yelp_dataset, 'validation', max_rows=None)
yelp_test_df = display_dataset_as_table('yelp_polarity', yelp_dataset, 'test', max_rows=50)

# AG News
agnews_train_df = display_dataset_as_table('ag_news', agnews_dataset, 'train', max_rows=None)
agnews_val_df = display_dataset_as_table('ag_news', agnews_dataset, 'validation', max_rows=None)
agnews_test_df = display_dataset_as_table('ag_news', agnews_dataset, 'test', max_rows=50)

# DBPedia
dbpedia_train_df = display_dataset_as_table('dbpedia_14', dbpedia_dataset, 'train', max_rows=None)
dbpedia_val_df = display_dataset_as_table('dbpedia_14', dbpedia_dataset, 'validation', max_rows=None)
dbpedia_test_df = display_dataset_as_table('dbpedia_14', dbpedia_dataset, 'test', max_rows=50)

print("\n" + "="*100)
print("✓ All datasets displayed!")
print("="*100)


SST2 - TRAIN SPLIT

Total records: 32



,label,verbalizer,text_preview,mlm_text_preview
0,0,bad,exactly how genteel and unsurprising the execution turns out to be ...,exactly how genteel and unsurprising the execution turns out to be It was <mask>....
1,0,bad,an all-time low for kevin costner ...,an all-time low for kevin costner It was <mask>....
2,0,bad,several cliched movie structures : ...,several cliched movie structures : It was <mask>....
3,0,bad,america 's knee-jerk moral sanctimony ...,america 's knee-jerk moral sanctimony It was <mask>....
4,0,bad,a dumb comedy ...,a dumb comedy It was <mask>....
5,0,bad,the plan to make enough into ` an inspiring tale of survival wrapped in the heart-pounding suspense ...,the plan to make enough into ` an inspiring tale of survival wrapped in the heart-pounding suspense of a stylish psychological thriller ' has flopped ...
6,0,bad,"proficient , dull ...","proficient , dull It was <mask>...."
7,0,bad,chitchat that only self-aware neurotics engage in ...,chitchat that only self-aware neurotics engage in It was <mask>....
8,0,bad,tries to cram too many ingredients into one small pot . ...,tries to cram too many ingredients into one small pot . It was <mask>....
9,0,bad,paper-thin characterizations ...,paper-thin characterizations It was <mask>....



SST2 - VALIDATION SPLIT

Total records: 32



,label,verbalizer,text_preview,mlm_text_preview
0,0,bad,bad imitation ...,bad imitation It was <mask>....
1,0,bad,"listless , witless , ...","listless , witless , It was <mask>...."
2,0,bad,gussied up with so many distracting special effects and visual party tricks that it 's not clear whe...,gussied up with so many distracting special effects and visual party tricks that it 's not clear whether we 're supposed to shriek It was <mask>....
3,0,bad,ca n't disguise the fact that it 's inauthentic at its core and that its story just is n't worth tel...,ca n't disguise the fact that it 's inauthentic at its core and that its story just is n't worth telling . It was <mask>....
4,0,bad,put through torture ...,put through torture It was <mask>....
5,0,bad,uncomfortably strained ...,uncomfortably strained It was <mask>....
6,0,bad,is that they 're stuck with a script that prevents them from firing on all cylinders . ...,is that they 're stuck with a script that prevents them from firing on all cylinders . It was <mask>....
7,0,bad,wobbly ...,wobbly It was <mask>....
8,0,bad,sure to ultimately disappoint the action fans who will be moved to the edge of their seats by the dy...,sure to ultimately disappoint the action fans who will be moved to the edge of their seats by the dynamic first act It was <mask>....
9,0,bad,freewheeling trash-cinema roots ...,freewheeling trash-cinema roots It was <mask>....



SST2 - TEST SPLIT

Total records: 872



,label,verbalizer,text_preview,mlm_text_preview
0,1,great,it 's a charming and often affecting journey . ...,it 's a charming and often affecting journey . It was <mask>....
1,0,bad,unflinchingly bleak and desperate ...,unflinchingly bleak and desperate It was <mask>....
2,1,great,allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmma...,allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmmaker . It was <mask>....
3,1,great,"the acting , costumes , music , cinematography and sound are all astounding given the production 's ...","the acting , costumes , music , cinematography and sound are all astounding given the production 's austere locales . It was <mask>...."
4,0,bad,"it 's slow -- very , very slow . ...","it 's slow -- very , very slow . It was <mask>...."
...,...,...,...,...
867,0,bad,has all the depth of a wading pool . ...,has all the depth of a wading pool . It was <mask>....
868,1,great,a movie with a real anarchic flair . ...,a movie with a real anarchic flair . It was <mask>....
869,0,bad,a subject like this should inspire reaction in its audience ; the pianist does not . ...,a subject like this should inspire reaction in its audience ; the pianist does not . It was <mask>....
870,0,bad,... is an arthritic attempt at directing by callie khouri . ...,... is an arthritic attempt at directing by callie khouri . It was <mask>....


... (15 outputs left out)

,label,verbalizer,text_preview,mlm_text_preview
0,0,Company,International Tea Co. Stores was a leading chain of grocers based in London. It was a constituent o...,Category: <mask> International Tea Co. Stores was a leading chain of grocers based in London. It was a constituent of the FT 30 index of leading comp...
1,0,Company,First Transit is a United States based subsidiary of FirstGroup. It provides contract public transi...,Category: <mask> First Transit is a United States based subsidiary of FirstGroup. It provides contract public transit and paratransit services transi...
2,0,Company,Shepherd and Wedderburn LLP is a leading commercial law firm headquartered in Edinburgh and with of...,Category: <mask> Shepherd and Wedderburn LLP is a leading commercial law firm headquartered in Edinburgh and with offices in Glasgow Aberdeen and Lon...
3,0,Company,Hulman & Company is a private family-owned company founded in 1850 by Francis Hulman as a wholesale...,Category: <mask> Hulman & Company is a private family-owned company founded in 1850 by Francis Hulman as a wholesale grocery tobacco and liquor store...
4,0,Company,Brashs was one of Australia's biggest music retail chains specializing in the retail of various vin...,Category: <mask> Brashs was one of Australia's biggest music retail chains specializing in the retail of various vinyl records and CDs blank audio ca...
5,0,Company,Science Applications International Corporation (SAIC) is a US company headquartered in McLean Virgi...,Category: <mask> Science Applications International Corporation (SAIC) is a US company headquartered in McLean Virginia that provides government serv...
6,0,Company,The Open Geospatial Consortium (OGC) an international voluntary consensus standards organization or...,Category: <mask> The Open Geospatial Consortium (OGC) an international voluntary consensus standards organization originated in 1994. In the OGC more...
7,0,Company,AET Tanker Holdings Sdn Bhd formerly American Eagle Tankers is a Malaysian-owned global shipping co...,Category: <mask> AET Tanker Holdings Sdn Bhd formerly American Eagle Tankers is a Malaysian-owned global shipping company specialising in the ocean t...
8,0,Company,InterCity Group (NZ) Ltd is a passenger transport and tourism company and operates New Zealand's la...,Category: <mask> InterCity Group (NZ) Ltd is a passenger transport and tourism company and operates New Zealand's largest coach network and ferries a...
9,0,Company,TechnoMarine was founded in 1997 by French entrepreneur Franck Dubarry in Geneva with the launch of...,Category: <mask> TechnoMarine was founded in 1997 by French entrepreneur Franck Dubarry in Geneva with the launch of the Raft a chronograph on a tran...



DBPEDIA_14 - TEST SPLIT

Total records: 70000



,label,verbalizer,text_preview,mlm_text_preview
0,0,Company,TY KU /taɪkuː/ is an American alcoholic beverage company that specializes in sake and other spirits...,Category: <mask> TY KU /taɪkuː/ is an American alcoholic beverage company that specializes in sake and other spirits. The privately-held company was ...
1,0,Company,OddLot Entertainment founded in 2001 by longtime producers Gigi Pritzker and Deborah Del Prete (The...,Category: <mask> OddLot Entertainment founded in 2001 by longtime producers Gigi Pritzker and Deborah Del Prete (The Wedding Planner) is a film produ...
2,0,Company,Henkel AG & Company KGaA operates worldwide with leading brands and technologies in three business ...,Category: <mask> Henkel AG & Company KGaA operates worldwide with leading brands and technologies in three business areas: Laundry & Home Care Beauty...
3,0,Company,The GOAT Store (Games Of All Type Store) LLC is one of the largest retro gaming online stores and a...,Category: <mask> The GOAT Store (Games Of All Type Store) LLC is one of the largest retro gaming online stores and an Independent Video Game Publishi...
4,0,Company,RagWing Aircraft Designs (also called the RagWing Aeroplane Company and RagWing Aviation) was an Am...,Category: <mask> RagWing Aircraft Designs (also called the RagWing Aeroplane Company and RagWing Aviation) was an American aircraft design and manufa...
...,...,...,...,...
69995,13,Written,Energy Victory: Winning the War on Terror by Breaking Free of Oil is a 2007 book by Robert Zubrin. ...,Category: <mask> Energy Victory: Winning the War on Terror by Breaking Free of Oil is a 2007 book by Robert Zubrin. Zubrin's central argument is that...
69996,13,Written,Bestiario is a book of 8 short stories written by Julio Cortázar....,Category: <mask> Bestiario is a book of 8 short stories written by Julio Cortázar....
69997,13,Written,Wuthering Heights is a novel by Emily Brontë written between October 1845 and June 1846 and publish...,Category: <mask> Wuthering Heights is a novel by Emily Brontë written between October 1845 and June 1846 and published in 1847 under the pseudonym El...
69998,13,Written,L'Indépendant is a newspaper published in Luxembourg from 1945....,Category: <mask> L'Indépendant is a newspaper published in Luxembourg from 1945....



✓ All datasets displayed!


In [ ]:
def tokenize_and_prepare_for_mlm(examples, tokenizer, config, max_length=256):

    tokenized = tokenizer(
        examples['mlm_text'],
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors=None
    )

    # Finding mask token position and create verbalizer token ids
    mask_token_id = tokenizer.mask_token_id
    verbalizer_ids = []
    mask_positions = []

    for i, input_ids in enumerate(tokenized['input_ids']):

        try:
            mask_pos = input_ids.index(mask_token_id)
            mask_positions.append(mask_pos)
        except ValueError:

            mask_positions.append(-1)


        label = examples['label'][i]
        verbalizer_text = config['verbalizers'][label]


        verbalizer_tokens = tokenizer(verbalizer_text, add_special_tokens=False)['input_ids']


        if len(verbalizer_tokens) > 0:
            verbalizer_ids.append(verbalizer_tokens[0])
        else:
            verbalizer_ids.append(-100)


    tokenized['mask_position'] = mask_positions
    tokenized['verbalizer_id'] = verbalizer_ids
    tokenized['labels'] = examples['label']

    return tokenized


print("✓ Tokenization function defined")

✓ Tokenization function defined


In [ ]:
print("="*70)
print("TOKENIZING DATASETS")
print("="*70)


MAX_LENGTH = 256

# Store tokenized datasets
tokenized_datasets = {}

datasets_to_tokenize = {
    'sst2': sst2_dataset,
    'yelp_polarity': yelp_dataset,
    'ag_news': agnews_dataset,
    'dbpedia_14': dbpedia_dataset
}

for dataset_name, dataset in datasets_to_tokenize.items():
    print(f"\n{'─'*70}")
    print(f"Tokenizing {dataset_name.upper()}")
    print(f"{'─'*70}")

    config = DATASET_CONFIG[dataset_name]

    # Tokenize all splits
    tokenized_train = dataset['train'].map(
        lambda examples: tokenize_and_prepare_for_mlm(examples, tokenizer, config, MAX_LENGTH),
        batched=True,
        batch_size=32,
        remove_columns=dataset['train'].column_names  # Remove original text columns
    )

    tokenized_val = dataset['validation'].map(
        lambda examples: tokenize_and_prepare_for_mlm(examples, tokenizer, config, MAX_LENGTH),
        batched=True,
        batch_size=32,
        remove_columns=dataset['validation'].column_names
    )

    tokenized_test = dataset['test'].map(
        lambda examples: tokenize_and_prepare_for_mlm(examples, tokenizer, config, MAX_LENGTH),
        batched=True,
        batch_size=32,
        remove_columns=dataset['test'].column_names
    )


    tokenized_dataset = DatasetDict({
        'train': tokenized_train,
        'validation': tokenized_val,
        'test': tokenized_test
    })

    tokenized_datasets[dataset_name] = tokenized_dataset

    print(f"✓ {dataset_name} tokenized")
    print(f"  Train: {len(tokenized_train)} samples")
    print(f"  Validation: {len(tokenized_val)} samples")
    print(f"  Test: {len(tokenized_test)} samples")
    print(f"  Columns: {tokenized_train.column_names}")

print("\n" + "="*70)
print("✓ All datasets tokenized!")
print("="*70)

TOKENIZING DATASETS

──────────────────────────────────────────────────────────────────────
Tokenizing SST2
──────────────────────────────────────────────────────────────────────


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

✓ sst2 tokenized
  Train: 32 samples
  Validation: 32 samples
  Test: 872 samples
  Columns: ['input_ids', 'attention_mask', 'mask_position', 'verbalizer_id', 'labels']

──────────────────────────────────────────────────────────────────────
Tokenizing YELP_POLARITY
──────────────────────────────────────────────────────────────────────


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

... (7 outputs left out)

Map:   0%|          | 0/224 [00:00<?, ? examples/s]

Map:   0%|          | 0/224 [00:00<?, ? examples/s]

Map:   0%|          | 0/70000 [00:00<?, ? examples/s]

✓ dbpedia_14 tokenized
  Train: 224 samples
  Validation: 224 samples
  Test: 70000 samples
  Columns: ['input_ids', 'attention_mask', 'mask_position', 'verbalizer_id', 'labels']

✓ All datasets tokenized!


In [ ]:
# Inspecting a tokenized sample to verify everything is correct
print("="*70)
print("TOKENIZED DATA INSPECTION")
print("="*70)

for dataset_name, dataset in tokenized_datasets.items():
    print(f"\n{'─'*70}")
    print(f"{dataset_name.upper()} - Sample from Train")
    print(f"{'─'*70}")

    sample = dataset['train'][0]

    print(f"\nInput IDs (first 20): {sample['input_ids'][:20]}")
    print(f"Attention Mask (first 20): {sample['attention_mask'][:20]}")
    print(f"Mask Position: {sample['mask_position']}")
    print(f"Verbalizer ID: {sample['verbalizer_id']}")
    print(f"Label: {sample['labels']}")

    decoded_text = tokenizer.decode(sample['input_ids'])
    print(f"\nDecoded text: {decoded_text}")


    verbalizer_token = tokenizer.decode([sample['verbalizer_id']])
    print(f"Verbalizer token: '{verbalizer_token}'")


    if sample['mask_position'] != -1:
        mask_token = sample['input_ids'][sample['mask_position']]
        print(f"Token at mask position: {tokenizer.decode([mask_token])} (should be <mask>)")

    print()

TOKENIZED DATA INSPECTION

──────────────────────────────────────────────────────────────────────
SST2 - Sample from Train
──────────────────────────────────────────────────────────────────────

Input IDs (first 20): [0, 3463, 45548, 141, 821, 8530, 523, 8, 36637, 21434, 5, 7356, 4072, 66, 7, 28, 1437, 85, 21, 50264]
Attention Mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Mask Position: 19
Verbalizer ID: 10999
Label: 0

Decoded text: <s>exactly how genteel and unsurprising the execution turns out to be  It was<mask>.</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><p

In [ ]:
# Save tokenized datasets to Google Drive
print("="*70)
print("SAVING TOKENIZED DATASETS")
print("="*70)

tokenized_save_dir = os.path.join(SAVE_DIR, 'datasets_tokenized')
os.makedirs(tokenized_save_dir, exist_ok=True)

for dataset_name, dataset in tokenized_datasets.items():
    save_path = os.path.join(tokenized_save_dir, f'{dataset_name}_{n_shot}shot_tokenized')
    print(f"\nSaving {dataset_name} to: {save_path}")

    dataset.save_to_disk(save_path)


    tokenizer_info = {
        'tokenizer_name': 'roberta-base',
        'max_length': MAX_LENGTH,
        'vocab_size': tokenizer.vocab_size,
        'mask_token': tokenizer.mask_token,
        'mask_token_id': tokenizer.mask_token_id
    }

    with open(os.path.join(save_path, 'tokenizer_info.json'), 'w') as f:
        json.dump(tokenizer_info, f, indent=2)

    print(f"✓ {dataset_name} saved")

print("\n" + "="*70)
print("✓ All tokenized datasets saved!")
print(f"Location: {tokenized_save_dir}")
print("="*70)

print("\n--- How to load tokenized datasets later ---")
print("from datasets import load_from_disk")
print("dataset = load_from_disk('/content/drive/MyDrive/FYP_GradientFree_LoRA/datasets_tokenized/sst2_16shot_tokenized')")

SAVING TOKENIZED DATASETS

Saving sst2 to: /content/drive/MyDrive/FYP_GradientFree_LoRA/datasets_tokenized/sst2_16shot_tokenized


Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/872 [00:00<?, ? examples/s]

✓ sst2 saved

Saving yelp_polarity to: /content/drive/MyDrive/FYP_GradientFree_LoRA/datasets_tokenized/yelp_polarity_16shot_tokenized


Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

... (7 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/70000 [00:00<?, ? examples/s]

✓ dbpedia_14 saved

✓ All tokenized datasets saved!
Location: /content/drive/MyDrive/FYP_GradientFree_LoRA/datasets_tokenized

--- How to load tokenized datasets later ---
from datasets import load_from_disk
dataset = load_from_disk('/content/drive/MyDrive/FYP_GradientFree_LoRA/datasets_tokenized/sst2_16shot_tokenized')


MODULE 2 & 3

In [ ]:
import torch
import torch.nn as nn

class SoftPromptEmbedding(nn.Module):
    def __init__(self, base_model, prompt_length=20):
        super().__init__()
        self.base_model = base_model
        self.prompt_length = prompt_length
        self.hidden_size = base_model.config.hidden_size

        # Initialize prompt embeddings
        self.soft_prompt = nn.Parameter(
            torch.randn(prompt_length, self.hidden_size) * 0.02
        )

    def forward(self, input_ids, attention_mask=None, labels=None):
        batch_size = input_ids.size(0)

        # Get token embeddings from model
        inputs_embeds = self.base_model.get_input_embeddings()(input_ids)

        # Expand soft prompt for batch
        prompt_embed = self.soft_prompt.unsqueeze(0).expand(
            batch_size, -1, -1
        )

        # Concatenate prompt + input embeddings
        inputs_embeds = torch.cat([prompt_embed, inputs_embeds], dim=1)

        # Adjust attention mask
        if attention_mask is not None:
            prompt_mask = torch.ones(
                batch_size, self.prompt_length,
                device=attention_mask.device
            )
            attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        outputs = self.base_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels
        )

        return outputs

In [ ]:
def create_projection_matrix(intrinsic_dim, full_dim, device):
    return torch.randn(full_dim, intrinsic_dim, device=device) / intrinsic_dim

In [ ]:
def evaluate_prompt_vector(
    model,
    intrinsic_vector,
    projection_matrix,
    train_loader,
    device
):
    model.eval()

    prompt_vector = projection_matrix @ intrinsic_vector
    prompt_vector = prompt_vector.view(
        model.prompt_length,
        model.hidden_size
    )

    model.soft_prompt.data = prompt_vector

    total_loss = 0
    total_samples = 0

    criterion = torch.nn.CrossEntropyLoss(reduction='sum')

    with torch.no_grad():
        for batch in train_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            mask_positions = batch["mask_position"]
            verbalizer_ids = batch["verbalizer_id"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits  # (batch, seq_len, vocab)

            batch_size = input_ids.size(0)

            for i in range(batch_size):

                mask_pos = mask_positions[i].item()
                if mask_pos == -1:
                    continue

                mask_logits = logits[i, mask_pos]  # (vocab_size,)

                # verbalizer_ids[i] is class index → we need actual token id
                target = verbalizer_ids[i].unsqueeze(0)

                loss = criterion(
                    mask_logits.unsqueeze(0),
                    target
                )

                total_loss += loss.item()
                total_samples += 1

    if total_samples == 0:
        return float("inf")

    return total_loss / total_samples

In [ ]:
!pip install cma -q
!pip install scikit-learn -q
!pip install psutil -q  # For memory tracking

print("✓ Optimization libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 10.0 MB/s eta 0:00:00
✓ Optimization libraries installed


In [ ]:
import time
import cma
import numpy as np

def optimize_prompt_with_cma(
    model,
    train_loader,
    intrinsic_dim,
    device,
    budget=5000,
    sigma=0.1,
    popsize=16
):
    print("\n" + "="*60)
    print("STARTING BBT PROMPT OPTIMIZATION (CMA-ES)")
    print("="*60)

    full_dim = model.prompt_length * model.hidden_size

    projection_matrix = create_projection_matrix(
        intrinsic_dim,
        full_dim,
        device
    )

    es = cma.CMAEvolutionStrategy(
        intrinsic_dim * [0],
        sigma,
        {'popsize': popsize, 'verbose': -1}
    )

    best_loss = float("inf")
    start_time = time.time()
    iteration = 0

    while not es.stop() and es.countevals < budget:

        iteration += 1
        iter_start = time.time()

        solutions = es.ask()
        losses = []

        for sol in solutions:
            z = torch.tensor(sol, device=device, dtype=torch.float32)

            loss = evaluate_prompt_vector(
                model,
                z,
                projection_matrix,
                train_loader,
                device
            )

            losses.append(loss)

        es.tell(solutions, losses)

        current_best = min(losses)
        avg_loss = sum(losses) / len(losses)

        if current_best < best_loss:
            best_loss = current_best

        iter_time = time.time() - iter_start
        total_time = time.time() - start_time

        print(f"Iter {iteration:03d} | "
              f"Best: {best_loss:.4f} | "
              f"Avg: {avg_loss:.4f} | "
              f"Evals: {es.countevals}/{budget} | "
              f"IterTime: {iter_time:.2f}s | "
              f"TotalTime: {total_time:.2f}s")

    print("\nOptimization Complete.")
    print(f"Final Best Loss: {best_loss:.4f}")
    print("="*60)

    best_z = torch.tensor(
        es.result.xbest,
        device=device,
        dtype=torch.float32
    )

    return best_z, projection_matrix

In [ ]:
def train_blackbox_prompt_tuning(
    base_model,
    train_loader,
    intrinsic_dim=100,
    prompt_length=20,
    device="cuda"
):
    base_model.eval()
    for param in base_model.parameters():
        param.requires_grad = False

    model = SoftPromptEmbedding(
        base_model,
        prompt_length=prompt_length
    ).to(device)

    best_z, projection_matrix = optimize_prompt_with_cma(
        model,
        train_loader,
        intrinsic_dim,
        device
    )

    # Set final prompt
    full_dim = prompt_length * model.hidden_size
    prompt_vector = projection_matrix @ best_z
    prompt_vector = prompt_vector.view(
        prompt_length,
        model.hidden_size
    )

    model.soft_prompt.data = prompt_vector

    return model

In [ ]:
from transformers import RobertaForMaskedLM

model = RobertaForMaskedLM.from_pretrained("roberta-base")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print("✓ RoBERTaForMaskedLM loaded for BBT")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ RoBERTaForMaskedLM loaded for BBT


MODULE 4

In [ ]:
class FireworksAlgorithm:

    def __init__(self, dimension, population_size=10, m_sparks=30, a=0.04, b=0.8, bounds=None):
        self.dimension = dimension
        self.population_size = population_size
        self.m_sparks = m_sparks
        self.a = a
        self.b = b
        self.bounds = bounds if bounds is not None else (-1, 1)

        # Initialize population
        self.population = np.random.randn(population_size, dimension) * 0.1
        self.fitness = np.full(population_size, np.inf)
        self.best_solution = None
        self.best_fitness = np.inf
        self.generation = 0


        self.no_improvement_count = 0

    def explode(self, firework_idx, amplitude):


        num_sparks = max(2, min(self.m_sparks // 4, 8))
        sparks = []

        for _ in range(num_sparks):
            spark = self.population[firework_idx].copy()

            num_dims = max(1, self.dimension // 10)
            dims = np.random.choice(self.dimension, num_dims, replace=False)

            displacement = np.random.uniform(-1, 1, num_dims) * amplitude
            spark[dims] += displacement
            spark = np.clip(spark, self.bounds[0], self.bounds[1])
            sparks.append(spark)

        return np.array(sparks)

    def gaussian_spark(self, firework_idx):

        spark = self.population[firework_idx].copy()
        num_dims = max(1, self.dimension // 10)
        dims = np.random.choice(self.dimension, num_dims, replace=False)
        spark[dims] *= np.random.normal(1.0, 0.1, num_dims)
        spark = np.clip(spark, self.bounds[0], self.bounds[1])
        return spark

    def ask(self):

        candidates = []


        num_explode = max(2, self.population_size // 2)
        best_indices = np.argsort(self.fitness)[:num_explode]

        for idx in best_indices:

            if np.isfinite(self.fitness[idx]) and np.isfinite(self.best_fitness):
                fitness_range = max(np.max(self.fitness[np.isfinite(self.fitness)]) - self.best_fitness, 1e-10)
                amplitude = self.a * max(self.fitness[idx] - self.best_fitness, 0) / fitness_range
                amplitude = np.clip(amplitude, 0.01, 1.0)
            else:
                amplitude = 0.5

            sparks = self.explode(idx, amplitude)
            candidates.extend(sparks)


        for _ in range(2):
            idx = np.random.choice(best_indices)
            candidates.append(self.gaussian_spark(idx))

        return np.array(candidates)

    def tell(self, candidates, fitness_values):


        all_solutions = np.vstack([self.population, candidates])
        all_fitness = np.concatenate([self.fitness, fitness_values])


        valid_mask = np.isfinite(all_fitness)
        all_solutions = all_solutions[valid_mask]
        all_fitness = all_fitness[valid_mask]

        if len(all_fitness) < self.population_size:

            needed = self.population_size - len(all_solutions)
            new_solutions = np.random.randn(needed, self.dimension) * 0.1
            all_solutions = np.vstack([all_solutions, new_solutions])
            all_fitness = np.concatenate([all_fitness, np.full(needed, np.inf)])

        # Select best
        sorted_indices = np.argsort(all_fitness)[:self.population_size]
        self.population = all_solutions[sorted_indices]
        self.fitness = all_fitness[sorted_indices]

        # Update best
        old_best = self.best_fitness
        if self.fitness[0] < self.best_fitness:
            self.best_fitness = self.fitness[0]
            self.best_solution = self.population[0].copy()
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1

        self.generation += 1

        return self.best_solution, self.best_fitness


print("✓ FIXED Fireworks Algorithm implemented")

✓ FIXED Fireworks Algorithm implemented


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

def evaluate_model(model, dataloader, device):
    """
    Evaluate MaskedLM model using verbalizer tokens at <mask> position.
    Works for BBT-style soft prompt tuning.
    """

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            mask_positions = batch['mask_position']
            verbalizer_ids = batch['verbalizer_id']  # shape: (batch, num_classes)
            labels = batch['labels']

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits  # (batch, seq_len, vocab_size)

            batch_size = input_ids.size(0)

            for i in range(batch_size):

                mask_pos = mask_positions[i].item()

                if mask_pos == -1:
                    mask_pos = 0

                # Get logits at mask position
                mask_logits = logits[i, mask_pos]  # (vocab_size,)

                # Get logits only for verbalizer tokens
                class_logits = mask_logits[verbalizer_ids[i].to(device)]

                # Pick class with highest score
                prediction = torch.argmax(class_logits).item()

                all_predictions.append(prediction)
                all_labels.append(labels[i].item())

    accuracy = accuracy_score(all_labels, all_predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average='weighted',
        zero_division=0
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
from torch.utils.data import DataLoader

def create_dataloaders(tokenized_datasets, batch_size=8):
    """
    Creating DataLoaders for train, validation, and test sets
    """
    def collate_fn(examples):

        batch = {
            'input_ids': torch.tensor([ex['input_ids'] for ex in examples], dtype=torch.long),
            'attention_mask': torch.tensor([ex['attention_mask'] for ex in examples], dtype=torch.long),
            'mask_position': torch.tensor([ex['mask_position'] for ex in examples], dtype=torch.long),
            'verbalizer_id': torch.tensor([ex['verbalizer_id'] for ex in examples], dtype=torch.long),
            'labels': torch.tensor([ex['labels'] for ex in examples], dtype=torch.long)
        }
        return batch

    dataloaders = {}

    for dataset_name, dataset in tokenized_datasets.items():
        dataloaders[dataset_name] = {
            'train': DataLoader(
                dataset['train'],
                batch_size=batch_size,
                shuffle=True,
                collate_fn=collate_fn
            ),
            'validation': DataLoader(
                dataset['validation'],
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_fn
            ),
            'test': DataLoader(
                dataset['test'],
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_fn
            )
        }

    return dataloaders


print("✓ DataLoader creation function defined")

✓ DataLoader creation function defined


In [ ]:
from transformers import RobertaForMaskedLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_model = RobertaForMaskedLM.from_pretrained("roberta-base")
base_model = base_model.to(device)

print("✓ RoBERTaForMaskedLM loaded")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ RoBERTaForMaskedLM loaded


In [ ]:
dataloaders = create_dataloaders(tokenized_datasets, batch_size=8)

train_loader = dataloaders["sst2"]["train"]
val_loader = dataloaders["sst2"]["validation"]
test_loader = dataloaders["sst2"]["test"]

print("✓ Dataloaders ready")

✓ Dataloaders ready


In [ ]:
trained_model = train_blackbox_prompt_tuning(
    base_model,
    train_loader,
    intrinsic_dim=500,
    prompt_length=50,
    device=device
)

print("✓ BBT Training Complete")


STARTING BBT PROMPT OPTIMIZATION (CMA-ES)
Iter 001 | Best: 14.3097 | Avg: 14.5650 | Evals: 16/5000 | IterTime: 11.37s | TotalTime: 11.37s
Iter 002 | Best: 14.0372 | Avg: 14.3479 | Evals: 32/5000 | IterTime: 11.55s | TotalTime: 22.92s
Iter 003 | Best: 13.9393 | Avg: 14.1555 | Evals: 48/5000 | IterTime: 11.90s | TotalTime: 34.82s
Iter 004 | Best: 13.7690 | Avg: 13.9600 | Evals: 64/5000 | IterTime: 12.44s | TotalTime: 47.26s
Iter 005 | Best: 13.6210 | Avg: 13.8509 | Evals: 80/5000 | IterTime: 12.96s | TotalTime: 60.22s
Iter 006 | Best: 13.3474 | Avg: 13.6775 | Evals: 96/5000 | IterTime: 12.63s | TotalTime: 72.86s
Iter 007 | Best: 13.2851 | Avg: 13.4716 | Evals: 112/5000 | IterTime: 12.11s | TotalTime: 84.97s
Iter 008 | Best: 13.2030 | Avg: 13.3815 | Evals: 128/5000 | IterTime: 11.82s | TotalTime: 96.79s
Iter 009 | Best: 13.0804 | Avg: 13.2375 | Evals: 144/5000 | IterTime: 11.86s | TotalTime: 108.65s
Iter 010 | Best: 12.9108 | Avg: 13.1137 | Evals: 160/5000 | IterTime: 12.07s | TotalTime:

In [ ]:
val_metrics = evaluate_model(trained_model, val_loader, device)
print("Validation:", val_metrics)

Validation: {'accuracy': 0.5, 'precision': 0.25, 'recall': 0.5, 'f1': 0.3333333333333333}


In [ ]:
test_metrics = evaluate_model(trained_model, test_loader, device)
print("Test:", test_metrics)

Test: {'accuracy': 0.4908256880733945, 'precision': 0.24090985607272117, 'recall': 0.4908256880733945, 'f1': 0.32318983768525056}
